# 8장 (실습 4): MCP가 MCP를 부를 때, 호출받는 쪽이 호출자를 확인하기

> 책 『AI 보안, 클라우드에서 시작한다』 8장 [실습] 4번을 노트북에서 단계별로 실행하는 버전입니다. Agent가 MCP Server A를 부르고, A가 다시 MCP Server B를 부르는 구조에서 **호출받는 쪽이 "어느 Client가, 누구를 대신해, 어떤 경로로 부르는가"를 확인하는 방법**을 코드로 봅니다. 표 8-9의 "인증·인가 부재"와 "토큰 전달과 혼동된 대리인"에 해당합니다. 소요 시간은 30분입니다.
> 같은 폴더의 [README.md](./README.md)와 내용이 같고, 셀을 위에서부터 차례로 실행하면 됩니다.

## 🎯 학습 목표
1. 토큰의 서명과 범위만 보는 MCP Server에서 **진짜 토큰으로도 일어나는 사고**(승인되지 않은 Agent, 토큰 도용, 토큰 그대로 넘기기, 중간 Server 건너뛰기)를 확인합니다.
2. 호출받는 쪽이 확인하는 네 가지(대상, 소유 증명, 호출 Client, 범위와 경로)가 **각 호출을 왜 막는지** 확인합니다.
3. 토큰 교환으로 A가 B를 부를 때 **범위가 좁아지고 위임 경로가 남는 것**을 토큰 내용으로 확인하고, 경로 검사를 직접 더해 봅니다.

## 📝 실습 구성

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | 예정 |
| 1 | 토큰만 확인하는 MCP Server | 예정 |
| 2 | 호출자까지 확인하는 MCP Server | 예정 |
| 3 | A가 B를 부를 때 무슨 일이 일어났나 | 예정 |
| 4 | 확인한 것과 확인하지 못한 것 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

## 실습 시나리오

김대리가 보고서 작성 Agent(`report-agent`)에게 고객 현황 요약을 맡깁니다. Agent는 사내 문서 MCP Server A(`docs-hub`)를 부르고, A는 고객 정보를 얻으려고 CRM MCP Server B(`crm-mcp`)를 다시 부릅니다. A는 Agent에게는 Server이고, B에게는 Client입니다.

![그림 실습8-4](../docs/images/그림실습8-4_MCP간_호출인증.png)

*[그림 실습8-4] 실습 4의 구조: A는 Agent에게는 Server, B에게는 Client다*

호출받는 쪽(A와 B)이 확인할 수 있는 것은 네 가지입니다.

`[표 실습8-7]` 호출받는 MCP Server가 확인하는 것

| 확인 | 질문 | 토큰과 요청에서 보는 곳 |
|---|---|---|
| ① 대상 | 이 토큰이 나에게 쓰라고 발급된 것인가 | 토큰의 `aud` |
| ② 소유 증명 | 토큰을 받은 Client가 직접 보냈는가, 아니면 토큰만 빼낸 누군가인가 | 토큰의 `cnf`와 요청마다 붙는 `DPoP` 증명 |
| ③ 호출 Client | 그 Client가 나를 부르도록 승인되었는가 | 토큰의 `azp`와 Server의 승인 목록 |
| ④ 범위와 경로 | 이 Tool을 쓸 권한이 있는가, 누구를 대신해 어떤 경로로 왔는가 | 토큰의 `scope`, `sub`, `act` |

MCP Server의 검사 수준을 두 가지로 바꿔 가며 같은 호출 6건을 보냅니다.

| 수준 | MCP Server가 확인하는 것 | A가 B를 부를 때 쓰는 토큰 |
|---|---|---|
| `bearer` | 서명·만료와 범위(④의 scope)만 | Agent에게 받은 토큰을 그대로 넘김(Token Passthrough) |
| `secure` | ①②③④ 모두 | A 자신의 신원으로 인가 서버에서 교환해 받은 B 전용 토큰 |

> 🔎 (용어 풀이)
> - **Bearer 토큰**: "가진 사람이 주인"인 토큰입니다. 현금처럼, 주운 사람도 그대로 쓸 수 있습니다.
> - **소유 증명(Proof of Possession)**: 토큰을 쓸 때마다 "이 토큰을 받은 Client의 키를 내가 갖고 있다"는 서명을 함께 보내는 방식입니다. 기명식 수표처럼, 토큰만 빼내서는 쓸 수 없습니다. 이 실습은 DPoP(RFC 9449)를 흉내 냈고, 서버끼리라면 mTLS 인증서에 토큰을 묶는 방식(RFC 8705)도 씁니다.
> - **client assertion**: Client가 인가 서버에 "나는 누구다"를 증명할 때 비밀번호(client secret) 대신 자기 키로 서명해 보내는 짧은 토큰입니다(private_key_jwt).
> - **azp와 act**: `azp`는 토큰을 받은 Client, `act`는 사용자를 대신해 움직인 행위자입니다. 위임이 여러 번 이어지면 `act` 안에 `act`가 겹쳐 쌓여 경로 전체가 남습니다(토큰 교환, RFC 8693).

> ✅ **실습 전 확인사항**
>
> **1. 실행 환경**<br>
> - 이 노트북은 **리눅스(WSL2의 Ubuntu 포함)** 환경을 가정합니다.
> - Python 3만 있으면 되고 외부 패키지, Docker, API Key가 필요 없습니다. 내 PC 안(127.0.0.1의 18080~18082 포트)에서 서비스 세 개를 띄웠다가 끝나면 내립니다. 다른 프로그램이 이 포트를 쓰고 있으면 실습이 실패하므로, 0장의 첫 셀에서 포트가 비어 있는지 먼저 확인합니다.
>
> **2. 노트북 열기**<br>
> 처음이라면 [SETUP.md](../SETUP.md)의 6단계(VS Code와 Jupyter 노트북 연결)를 먼저 한 번 해 둡니다. 그다음에는 아래 두 방법 가운데 하나로 이 노트북을 엽니다.
> - **VS Code**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs && code .`를 실행합니다. 창 왼쪽 아래에 `WSL: Ubuntu-24.04`가 보이면, 왼쪽 탐색기에서 `ch08-devsecops-image-gate` 폴더를 펼쳐 이 노트북을 엽니다. 그리고 오른쪽 위 커널 표시를 눌러 **Select Another Kernel...** → **Python Environments...** → `.venv-lab`을 고릅니다. 목록에 `.venv-lab`이 없으면 6단계 ⑥의 2번처럼 `Python: Select Interpreter` → **Enter interpreter path...**로 `~/.venv-lab/bin/python`의 위치를 먼저 알려 줍니다.
> - **웹 브라우저(Jupyter Lab)**: Ubuntu 터미널에서 `cd ~/ai-cloud-security-labs/ch08-devsecops-image-gate && ~/.venv-lab/bin/jupyter lab`을 실행하고, 화면에 나온 `http://localhost:8888/...` 주소를 Windows의 브라우저에서 엽니다.
> - 노트북 파일만 따로 내려받거나 다른 폴더로 옮기지 마세요. 노트북은 자기가 들어 있는 `ch08-devsecops-image-gate` 폴더의 실습 파일을 읽습니다. 위 두 방법처럼 저장소 안에서 열면 따로 신경 쓸 것은 없습니다.
>
> **3. 기준 버전**<br>
> - 이 노트북은 2026년 10월 25일 Ubuntu 24.04(WSL2), Python 3.12에서 끝까지 실행해 확인했습니다. 실행 결과가 저장된 버전은 `ch08_lab4_mcp_caller_auth_executed.ipynb`입니다.

## 0. 실습 준비와 실습 파일

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 1 | 토큰만 확인하는 MCP Server | 예정 |
| 2 | 호출자까지 확인하는 MCP Server | 예정 |
| 3 | A가 B를 부를 때 무슨 일이 일어났나 | 예정 |
| 4 | 확인한 것과 확인하지 못한 것 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

In [1]:
import json
import os
import shutil
import socket
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display

LAB = Path.cwd()          # 이 노트북이 있는 폴더


def sh(cmd):
    """셸 명령을 실행하고 표준 출력을 문자열로 돌려줍니다."""
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()


def table(header, rows):
    """머리글과 행 목록으로 마크다운 표를 그립니다."""
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    lines += ["| " + " | ".join(str(c) for c in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))


def port_free(port):
    with socket.socket() as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


VERSION_CMD = {
    "python3": "python3 --version",
}
for tool, cmd in VERSION_CMD.items():
    state = f"✅ {sh(cmd).splitlines()[0] if sh(cmd) else ''}" if shutil.which(tool) else "❌ 없음"
    print(f"{tool:10} {state}")
for port in (18080, 18081, 18082):
    print(f"포트 {port} {'✅ 비어 있음' if port_free(port) else '❌ 사용 중(그 프로그램을 먼저 끕니다)'}")

python3    ✅ Python 3.12.3
포트 18080 ✅ 비어 있음
포트 18081 ✅ 비어 있음
포트 18082 ✅ 비어 있음


| 파일 | 역할 |
|---|---|
| `mcp-auth/services.py` | 인가 서버, MCP A, MCP B와 Client 동작. MCP Server의 검사 수준은 `MODE`로 바뀜 |
| `mcp-auth/scenarios.json` | 호출 6건. 정상 1·2단계 호출과 공격이 섞여 있음 |
| `mcp-auth/lab.py` | 세 서비스를 띄우고 `scenarios.json`의 호출을 보낸 뒤 결과를 출력 |

> 실습을 위해 단순화한 점: 모든 서명이 HMAC(HS256)이라 Client의 키를 등록부(`CLIENT_KEYS`)에 함께 둡니다. 실제로는 각 Client가 개인키를 혼자 갖고 공개키만 인가 서버에 등록하며, 호출받는 쪽도 공개키로 확인합니다. 사용자가 로그인해 토큰을 받는 과정(Authorization Code + PKCE)과 MCP의 연결 절차(initialize)도 생략했습니다.

실습 파일을 `~/mcp-auth-lab`으로 복사하고, 보낼 호출 6건을 읽어 봅니다. `aud`는 Agent가 어느 Server용 토큰을 받아 왔는지, `prove`가 `false`이면 소유 증명 없이 토큰만 보낸다는 뜻입니다.

In [2]:
WORK = Path.home() / "mcp-auth-lab"
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(LAB / "mcp-auth", WORK)
SCENARIOS = json.loads((WORK / "scenarios.json").read_text(encoding="utf-8"))

table(["번호", "누가", "Client", "→ Server", "Tool", "토큰 aud", "소유 증명", "설명"],
      [(i, s["who"], s["client"], s["target"], s["tool"], s["aud"], "보냄" if s.get("prove", True) else "**안 보냄**", s["note"])
       for i, s in enumerate(SCENARIOS, 1)])

| 번호 | 누가 | Client | → Server | Tool | 토큰 aud | 소유 증명 | 설명 |
|---|---|---|---|---|---|---|---|
| 1 | 김대리 | report-agent | A | search_docs | A | 보냄 | 승인된 Agent가 A의 문서 검색 호출(정상, 1단계) |
| 2 | 김대리 | report-agent | A | crm_summary | A | 보냄 | A가 다시 B를 부르는 고객 요약(정상, 2단계) |
| 3 | 김대리 | shadow-agent | A | search_docs | A | 보냄 | 인가 서버에 등록은 됐지만 A에는 승인되지 않은 Agent |
| 4 | 공격자 | report-agent | A | search_docs | A | **안 보냄** | report-agent의 토큰만 빼내 재사용(키는 없어 소유 증명을 못 만듦) |
| 5 | 김대리 | report-agent | B | crm_read | A | 보냄 | A용 토큰으로 B를 직접 호출 |
| 6 | 김대리 | report-agent | B | crm_read | B | 보냄 | B용 토큰을 받아 A를 건너뛰고 B를 직접 호출 |

In [3]:
def lab(mode):
    """lab.py를 실행하고 출력을 돌려줍니다. 터미널의 python3 lab.py ...와 같습니다."""
    r = subprocess.run([sys.executable, "lab.py", mode], cwd=WORK, capture_output=True, text=True)
    return r.stdout + r.stderr

## 1. 토큰만 확인하는 MCP Server

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 토큰만 확인하는 MCP Server | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 2 | 호출자까지 확인하는 MCP Server | 예정 |
| 3 | A가 B를 부를 때 무슨 일이 일어났나 | 예정 |
| 4 | 확인한 것과 확인하지 못한 것 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

In [4]:
print(lab("bearer"))

MCP Server 검사 수준: bearer

1 report-agent → A search_docs 실행     사내 문서 검색 실행됨(사용자: kim)
  └ 김대리: 승인된 Agent가 A의 문서 검색 호출(정상, 1단계)
2 report-agent → A crm_summary 실행     MCP B 응답 ▶ 고객 3명 조회. B가 확인한 경로: kim → (확인 안 함), scope=crm:read docs:read
  └ 김대리: A가 다시 B를 부르는 고객 요약(정상, 2단계)
3 shadow-agent → A search_docs 실행     사내 문서 검색 실행됨(사용자: kim)
  └ 김대리: 인가 서버에 등록은 됐지만 A에는 승인되지 않은 Agent
4 report-agent → A search_docs 실행     사내 문서 검색 실행됨(사용자: kim)
  └ 공격자: report-agent의 토큰만 빼내 재사용(키는 없어 소유 증명을 못 만듦)
5 report-agent → B crm_read    실행     고객 3명 조회. B가 확인한 경로: kim → (확인 안 함), scope=crm:read docs:read
  └ 김대리: A용 토큰으로 B를 직접 호출
6 report-agent → B crm_read    실행     고객 3명 조회. B가 확인한 경로: kim → (확인 안 함), scope=crm:read docs:read
  └ 김대리: B용 토큰을 받아 A를 건너뛰고 B를 직접 호출



모든 호출이 실행되었습니다. 토큰은 모두 인가 서버가 서명한 진짜이고 범위도 맞기 때문입니다. 그런데 호출받는 쪽은 **김대리의 토큰이라는 것만 알고, 누가 그 토큰을 내밀었는지는 모릅니다.**

- **3번**: 승인되지 않은 Agent도 A를 쓸 수 있습니다. 회사가 승인하지 않은 Agent 도구(Shadow AI)가 사용자의 토큰만 얻으면 사내 MCP Server를 마음대로 부를 수 있습니다.
- **4번**: 토큰을 빼낸 공격자도 그대로 쓸 수 있습니다. Bearer 토큰은 가진 사람이 주인입니다.
- **2번과 5번**: A는 받은 토큰을 B에 그대로 넘겼고, B는 A용 토큰(5번)도 받아 줬습니다. 토큰 하나가 A와 B 모두에서 통하므로 하나가 새면 둘 다 열립니다. B가 받은 범위에는 고객 조회와 상관없는 `docs:read`까지 들어 있습니다.
- **6번**: Agent가 A를 건너뛰고 B를 바로 불렀습니다. A에 걸어 둔 통제(기록, 마스킹, 승인)를 모두 피해 갈 수 있습니다.
- **B가 확인한 경로**: 모두 `kim → (확인 안 함)`입니다. 사고가 나도 B의 기록만으로는 어느 Agent가 어느 Server를 거쳐 왔는지 알 수 없습니다.

## 2. 호출자까지 확인하는 MCP Server

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 토큰만 확인하는 MCP Server | ✅ 완료 |
| 2 | 호출자까지 확인하는 MCP Server | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 3 | A가 B를 부를 때 무슨 일이 일어났나 | 예정 |
| 4 | 확인한 것과 확인하지 못한 것 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

In [5]:
print(lab("secure"))

MCP Server 검사 수준: secure

1 report-agent → A search_docs 실행     사내 문서 검색 실행됨(사용자: kim)
  └ 김대리: 승인된 Agent가 A의 문서 검색 호출(정상, 1단계)
2 report-agent → A crm_summary 실행     MCP B 응답 ▶ 고객 3명 조회. B가 확인한 경로: kim → report-agent → docs-hub, scope=crm:read
  └ 김대리: A가 다시 B를 부르는 고객 요약(정상, 2단계)
3 shadow-agent → A search_docs 403 거부 이 Server에 승인되지 않은 Client: shadow-agent
  └ 김대리: 인가 서버에 등록은 됐지만 A에는 승인되지 않은 Agent
4 report-agent → A search_docs 401 거부 소유 증명 없음(토큰만 가진 쪽)
  └ 공격자: report-agent의 토큰만 빼내 재사용(키는 없어 소유 증명을 못 만듦)
5 report-agent → B crm_read    401 거부 이 Server용 토큰이 아님(aud=https://docs-hub.hanbit.example/mcp)
  └ 김대리: A용 토큰으로 B를 직접 호출
6 report-agent → B crm_read    403 거부 이 Server에 승인되지 않은 Client: report-agent
  └ 김대리: B용 토큰을 받아 A를 건너뛰고 B를 직접 호출



### 2.1. 호출별로 무엇이 막았나

이번에는 `services.py`를 노트북에서 직접 불러 세 서비스를 띄워 둡니다. `services.MODE`만 바꾸면 같은 서비스의 검사 수준이 바뀌므로, 두 수준의 결과를 한 표에 놓고 거부 사유가 어느 확인에서 나왔는지 봅니다.

> 노트북 안에서 띄운 서비스는 6장 정리 셀에서 내립니다. 그 전에 위의 `lab()`을 다시 실행하면 포트가 겹쳐 실패합니다.

In [6]:
sys.path.insert(0, str(WORK))
import services as sv

SERVERS = sv.start()
AUD = {"A": sv.A_RESOURCE, "B": sv.B_RESOURCE}
TARGET = {"A": sv.A_URL + "/mcp", "B": sv.B_URL + "/mcp"}
MSG = {"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": {"name": "search_docs", "arguments": {}}}


def run_scenarios(mode, scenarios=SCENARIOS):
    """lab.py의 main()과 같은 호출을 보내고 (판정, 사유) 목록을 돌려줍니다."""
    sv.MODE = mode
    rows = []
    for sc in scenarios:
        client = sv.Client(sc["client"])
        _, body = client.get_token("kim", AUD[sc["aud"]], "docs:read crm:read")
        prove = mode == "secure" and sc.get("prove", True)
        status, body = client.call(TARGET[sc["target"]], body["access_token"], sc["tool"], prove)
        if status != 200:
            rows.append((f"{status} 거부", body["error"]))
        else:
            result = body["result"]
            rows.append(("실패" if result["isError"] else "실행", result["content"][0]["text"]))
    return rows


STAGE = [("이 Server용 토큰이 아님", "① 대상"), ("소유 증명", "② 소유 증명"), ("재사용된 증명", "② 소유 증명"),
         ("승인되지 않은 Client", "③ 호출 Client"), ("범위", "④ 범위")]


def stage_of(reason):
    return next((s for key, s in STAGE if key in reason), "-")


bearer, secure = run_scenarios("bearer"), run_scenarios("secure")
table(["번호", "호출", "bearer", "secure", "막은 확인", "secure 사유"],
      [(i, sc["note"], b[0], s[0], stage_of(s[1]) if s[0] != "실행" else "-", s[1])
       for i, (sc, b, s) in enumerate(zip(SCENARIOS, bearer, secure), 1)])

| 번호 | 호출 | bearer | secure | 막은 확인 | secure 사유 |
|---|---|---|---|---|---|
| 1 | 승인된 Agent가 A의 문서 검색 호출(정상, 1단계) | 실행 | 실행 | - | 사내 문서 검색 실행됨(사용자: kim) |
| 2 | A가 다시 B를 부르는 고객 요약(정상, 2단계) | 실행 | 실행 | - | MCP B 응답 ▶ 고객 3명 조회. B가 확인한 경로: kim → report-agent → docs-hub, scope=crm:read |
| 3 | 인가 서버에 등록은 됐지만 A에는 승인되지 않은 Agent | 실행 | 403 거부 | ③ 호출 Client | 이 Server에 승인되지 않은 Client: shadow-agent |
| 4 | report-agent의 토큰만 빼내 재사용(키는 없어 소유 증명을 못 만듦) | 실행 | 401 거부 | ② 소유 증명 | 소유 증명 없음(토큰만 가진 쪽) |
| 5 | A용 토큰으로 B를 직접 호출 | 실행 | 401 거부 | ① 대상 | 이 Server용 토큰이 아님(aud=https://docs-hub.hanbit.example/mcp) |
| 6 | B용 토큰을 받아 A를 건너뛰고 B를 직접 호출 | 실행 | 403 거부 | ③ 호출 Client | 이 Server에 승인되지 않은 Client: report-agent |

`[표 실습8-8]` 검사 수준별 결과와 막은 확인

| 호출 | bearer | secure | 막은 확인 |
|---|---|---|---|
| 3 승인되지 않은 Agent | 실행 | 403 | ③ 호출 Client(A의 승인 목록에 `shadow-agent` 없음) |
| 4 빼낸 토큰 재사용 | 실행 | 401 | ② 소유 증명 |
| 5 A용 토큰으로 B 호출 | 실행 | 401 | ① 대상 |
| 6 A를 건너뛰고 B 호출 | 실행 | 403 | ③ 호출 Client(B는 `docs-hub`만 허용) |

6번에 주목합니다. 토큰은 B용으로 정상 발급되었고 소유 증명도 맞습니다. 그래도 B는 "A를 거쳐서만 부를 수 있다"는 자기 규칙으로 막았습니다. **인가 서버가 토큰을 내줬다고 해서 호출받는 쪽이 판단을 넘겨서는 안 됩니다.** 어떤 Client를 받아 줄지는 각 Server가 정합니다.

### 2.2. 소유 증명은 요청 하나에만 쓸 수 있다

4번의 공격자는 토큰만 빼냈습니다. 그렇다면 **토큰과 소유 증명이 함께 담긴 요청을 통째로** 엿봤다면 어떨까요? 같은 토큰에 소유 증명을 여러 방식으로 붙여 A의 `search_docs`를 부릅니다.

In [7]:
sv.MODE = "secure"
agent = sv.Client("report-agent")
token = agent.get_token("kim", sv.A_RESOURCE, "docs:read")[1]["access_token"]
captured = agent.proof(TARGET["A"])            # 정상 요청에 붙었던 증명. 공격자가 이것까지 엿봤다고 가정

TRIES = [
    ("report-agent가 새로 만든 증명(정상)", captured),
    ("엿본 증명을 그대로 다시 붙임", captured),
    ("shadow-agent의 키로 만든 증명", sv.Client("shadow-agent").proof(TARGET["A"])),
    ("report-agent의 키로 만들었지만 B 주소용 증명", agent.proof(TARGET["B"])),
]
rows = []
for label, proof in TRIES:
    status, body = sv.http(TARGET["A"], MSG, {"Authorization": "Bearer " + token, "DPoP": proof})
    rows.append((label, status, body.get("error") or body["result"]["content"][0]["text"]))
table(["붙인 소유 증명", "상태", "결과"], rows)

| 붙인 소유 증명 | 상태 | 결과 |
|---|---|---|
| report-agent가 새로 만든 증명(정상) | 200 | 사내 문서 검색 실행됨(사용자: kim) |
| 엿본 증명을 그대로 다시 붙임 | 401 | 다른 주소용이거나 재사용된 증명 |
| shadow-agent의 키로 만든 증명 | 401 | 소유 증명이 토큰의 Client와 다름 |
| report-agent의 키로 만들었지만 B 주소용 증명 | 401 | 다른 주소용이거나 재사용된 증명 |

증명에는 보낼 주소(`htu`)와 한 번만 쓰는 번호(`jti`)가 들어 있고, 토큰의 `cnf`가 가리키는 Client의 키로 서명되어야 합니다. 그래서 요청을 엿봐도 **다시 보내거나, 다른 Server로 돌려쓰거나, 다른 Client가 대신 쓸 수 없습니다.** 실제 DPoP는 여기에 요청 시각(`iat`)과 토큰의 해시(`ath`)도 넣어 증명이 쓰일 수 있는 시간과 토큰까지 묶습니다.

> 🙋 (생각해 보기)
> 이 실습의 `SEEN_PROOFS`는 서버 메모리에 있는 집합이라, MCP Server가 여러 대로 늘어나면 서버끼리 공유되지 않습니다. 실무에서는 재사용 확인을 어디에 두어야 할까요? 증명의 유효 시간을 짧게 두는 것은 어떤 도움이 될까요?

## 3. A가 B를 부를 때 무슨 일이 일어났나

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 토큰만 확인하는 MCP Server | ✅ 완료 |
| 2 | 호출자까지 확인하는 MCP Server | ✅ 완료 |
| 3 | A가 B를 부를 때 무슨 일이 일어났나 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 4 | 확인한 것과 확인하지 못한 것 | 예정 |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

2번 호출에서 A는 받은 토큰①을 B에 넘기지 않고, 다음 순서로 B를 불렀습니다(`services.py`의 `crm_summary`).

1. A는 자기 키로 client assertion을 만들어 "나는 `docs-hub`다"를 인가 서버에 증명합니다.
2. 토큰①을 내고 "B용, `crm:read`만"이라고 토큰 교환을 요청합니다. 인가 서버는 토큰①이 정말 A에게 발급된 것인지 확인하고, 토큰①에 있던 범위 안에서만 내줍니다. 범위는 넓어질 수 없고 좁아지기만 합니다.
3. 새 토큰②에 A의 소유 증명을 붙여 B를 부릅니다.

같은 과정을 노트북에서 한 단계씩 실행해 두 토큰의 내용을 나란히 봅니다.

In [8]:
hub = sv.Client("docs-hub")
t1 = agent.get_token("kim", sv.A_RESOURCE, "docs:read crm:read")[1]["access_token"]   # Agent가 A용으로 받은 토큰①
status, body = hub.exchange(t1, sv.B_RESOURCE, "crm:read")                           # A가 B용으로 바꾼 토큰②
t2 = body["access_token"]
c1, c2 = sv.peek(t1), sv.peek(t2)

def claim(c, k):
    return f"`{json.dumps(c[k], ensure_ascii=False)}`" if k in c else "(없음)"


table(["클레임", "토큰① (Agent → A)", "토큰② (A → B)"],
      [(k, claim(c1, k), claim(c2, k)) for k in ("sub", "aud", "azp", "scope", "act", "cnf")])
print("토큰①의 경로:", " → ".join(sv.path(c1)))
print("토큰②의 경로:", " → ".join(sv.path(c2)))

| 클레임 | 토큰① (Agent → A) | 토큰② (A → B) |
|---|---|---|
| sub | `"kim"` | `"kim"` |
| aud | `"https://docs-hub.hanbit.example/mcp"` | `"https://crm-mcp.hanbit.example/mcp"` |
| azp | `"report-agent"` | `"docs-hub"` |
| scope | `"crm:read docs:read"` | `"crm:read"` |
| act | (없음) | `{"sub": "docs-hub", "act": {"sub": "report-agent"}}` |
| cnf | `{"kid": "report-agent"}` | `{"kid": "docs-hub"}` |

토큰①의 경로: kim → report-agent
토큰②의 경로: kim → report-agent → docs-hub


B는 토큰② 하나로 "김대리를 대신해 `report-agent`가 시작했고, `docs-hub`가 지금 부르고 있다"는 경로 전체와, 쓸 수 있는 범위가 `crm:read`뿐이라는 것을 압니다. 출력의 `kim → report-agent → docs-hub`가 이것입니다. 위임이 한 번 더 이어지면 `act` 안에 한 겹이 더 쌓입니다. 5-3에서 말한 "위임할 때마다 권한을 좁히고, 위임의 연쇄 전체를 기록한다"를 토큰으로 구현한 것입니다.

### 3.1. 인가 서버가 교환을 거절하는 경우

토큰 교환은 아무에게나 해 주지 않습니다. 다음 세 요청을 인가 서버에 보내 봅니다.

In [9]:
head, body_part, sig = t1.split(".")
forged_claims = dict(c1, scope="docs:read crm:read crm:write")      # 범위를 몰래 넓힌 토큰①
forged = ".".join([head, sv.b64(json.dumps(forged_claims).encode()), sig])

EXCHANGES = [
    ("A(docs-hub)가 자기가 받은 토큰①을 교환(정상)", hub, t1),
    ("Agent가 A를 건너뛰려고 토큰①을 직접 교환", agent, t1),
    ("A가 범위를 몰래 넓힌 토큰①을 교환", hub, forged),
]
rows = []
for label, who, subject in EXCHANGES:
    status, body = who.exchange(subject, sv.B_RESOURCE, "crm:read")
    result = f"scope={sv.peek(body['access_token'])['scope']}" if status == 200 else body["error_description"]
    rows.append((label, who.cid, status, result))
table(["교환 요청", "요청한 Client", "상태", "결과"], rows)

| 교환 요청 | 요청한 Client | 상태 | 결과 |
|---|---|---|---|
| A(docs-hub)가 자기가 받은 토큰①을 교환(정상) | docs-hub | 200 | scope=crm:read |
| Agent가 A를 건너뛰려고 토큰①을 직접 교환 | report-agent | 400 | 이 Client가 받은 토큰이 아님 |
| A가 범위를 몰래 넓힌 토큰①을 교환 | docs-hub | 400 | 서명 불일치 |

인가 서버는 ① 요청한 Client가 client assertion으로 자기를 증명했는지, ② 내민 토큰이 그 Client가 **받은** 토큰인지(토큰①의 `aud`가 A이므로 A만 바꿀 수 있음), ③ 토큰의 서명이 맞는지를 확인합니다. 그래서 Agent는 토큰①로 B용 토큰을 직접 만들어 A를 건너뛸 수 없고, 토큰 내용을 고쳐 범위를 넓힐 수도 없습니다.

## 4. 확인한 것과 확인하지 못한 것

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 토큰만 확인하는 MCP Server | ✅ 완료 |
| 2 | 호출자까지 확인하는 MCP Server | ✅ 완료 |
| 3 | A가 B를 부를 때 무슨 일이 일어났나 | ✅ 완료 |
| 4 | 확인한 것과 확인하지 못한 것 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 5 | 체크포인트와 과제 | 예정 |
| 6 | 정리 | 예정 |

| 항목 | 이 실습에서 | 실무에서 더 할 것 |
|---|---|---|
| Client 신원 증명 | 확인함: client assertion(키 서명) | 공개키 등록(private_key_jwt), 클라우드에서는 Workload Identity로 키 자체를 없앰(7-1) |
| 토큰 도용 방지 | 확인함: 소유 증명(DPoP 흉내) | 서버 사이는 mTLS 인증서 바인딩, 서비스 메시(SPIFFE)도 선택지 |
| 호출 Client 승인 | 확인함: Server별 승인 목록 | 승인 목록을 사내 레지스트리·MCP Gateway에서 한곳에 관리(5-3, 7-4) |
| 다단계 위임 | 확인함: 토큰 교환, 범위 축소, `act` 경로 | 위임 깊이 상한, 경로에 승인되지 않은 Client가 끼면 거부(과제 1) |
| 사용자 로그인 | 확인하지 않음 | Authorization Code + PKCE, MCP Client 등록 방식(사전 등록, Client ID 메타데이터 문서) |
| 기록 | 확인하지 않음 | `act` 경로와 Trace ID를 함께 남김(9장 실습 2) |

이 실습은 원격(HTTP) MCP Server의 경우입니다. 내 PC에서 실행되는 로컬(stdio) MCP Server는 네트워크 인증을 거치지 않고 사용자 권한으로 바로 실행되므로, 5장 실습 1처럼 접근 범위를 좁히고 API Key 같은 자격증명은 설정 파일에 그대로 적지 않고 OS의 키 저장소나 환경변수로 넘깁니다.

## 5. 체크포인트와 과제

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 토큰만 확인하는 MCP Server | ✅ 완료 |
| 2 | 호출자까지 확인하는 MCP Server | ✅ 완료 |
| 3 | A가 B를 부를 때 무슨 일이 일어났나 | ✅ 완료 |
| 4 | 확인한 것과 확인하지 못한 것 | ✅ 완료 |
| 5 | 체크포인트와 과제 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |
| 6 | 정리 | 예정 |

### 5.1. 체크포인트

- [ ] `bearer`에서 토큰이 모두 진짜였는데도 막아야 했던 호출은 무엇이었고, 각각 무엇을 확인해야 막을 수 있었나요?
- [ ] 6번처럼 B용으로 정상 발급된 토큰을 B가 거부한 이유를 설명할 수 있나요?
- [ ] A가 받은 토큰을 B에 그대로 넘기는 것과 토큰 교환을 하는 것은, B가 아는 정보와 받는 권한에서 어떻게 다른가요?
- [ ] 여러분 회사의 MCP Server는 "누가 부르는지"를 사용자 단위로만 보나요, Client(Agent·MCP) 단위로도 보나요?

### 5.2. 과제 1: B에 위임 경로 검사 더하기

지금의 B는 **지금 부르는 Client**(`azp`)만 봅니다. 그래서 A의 승인 목록이 느슨해져 `shadow-agent`가 A를 거쳐 들어오면, B는 "부른 쪽이 `docs-hub`니까 괜찮다"며 받아 줍니다. `act`에 경로 전체가 남아 있으므로 B는 **경로 전체**를 볼 수 있습니다.

`APPROVED_ROUTES`에 B가 받아 줄 경로(사용자를 뺀 Client 순서)를 적어, 아래 네 경우가 모두 기대대로 판정되게 만듭니다. 세 번째 경우는 중간 MCP Server(`summary-mcp`)가 하나 더 끼어 위임이 한 단계 더 이어진 토큰을 흉내 낸 것입니다.

In [10]:
APPROVED_ROUTES = [
    # TODO: B가 받아 줄 위임 경로를 Client 순서의 튜플로 적습니다.
]


def route_ok(claims):
    route = tuple(sv.path(claims)[1:])         # 사용자를 뺀 Client 경로
    return route in APPROVED_ROUTES


shadow = sv.Client("shadow-agent")
t_shadow = shadow.get_token("kim", sv.A_RESOURCE, "crm:read")[1]["access_token"]
CASES = [
    ("정상: Agent → A → B", sv.peek(t2), True),
    ("A의 승인 목록이 느슨해 shadow-agent가 A를 거쳐 옴",
     sv.peek(hub.exchange(t_shadow, sv.B_RESOURCE, "crm:read")[1]["access_token"]), False),
    ("중간 Server가 하나 더 낀 경로(흉내)",
     dict(c2, act={"sub": "docs-hub", "act": {"sub": "summary-mcp", "act": {"sub": "report-agent"}}}), False),
    ("Agent가 B용 토큰으로 직접", sv.peek(agent.get_token("kim", sv.B_RESOURCE, "crm:read")[1]["access_token"]), False),
]
table(["경우", "경로", "기대", "route_ok", "일치"],
      [(label, " → ".join(sv.path(c)), "허용" if want else "거부", "허용" if route_ok(c) else "거부",
        "✅" if route_ok(c) == want else "❌") for label, c, want in CASES])

| 경우 | 경로 | 기대 | route_ok | 일치 |
|---|---|---|---|---|
| 정상: Agent → A → B | kim → report-agent → docs-hub | 허용 | 거부 | ❌ |
| A의 승인 목록이 느슨해 shadow-agent가 A를 거쳐 옴 | kim → shadow-agent → docs-hub | 거부 | 거부 | ✅ |
| 중간 Server가 하나 더 낀 경로(흉내) | kim → report-agent → summary-mcp → docs-hub | 거부 | 거부 | ✅ |
| Agent가 B용 토큰으로 직접 | kim → report-agent | 거부 | 거부 | ✅ |

<details>
<summary>💡 힌트 보기</summary>

```python
APPROVED_ROUTES = [
    ("report-agent", "docs-hub"),
]
```

승인 목록(`allowed_clients`)이 "누가 나를 부를 수 있나"라면, 승인 경로는 "누가 누구를 거쳐 나를 부를 수 있나"입니다. 실제 B에 넣으려면 `McpServer.authenticate()`의 ③ 호출 Client 확인 바로 뒤에 `route_ok()` 검사를 더하고, 실패하면 403을 돌려줍니다. 경로를 하나하나 적기 어렵다면 "위임 깊이 상한(예: Client 3개까지)"과 "경로의 모든 Client가 사내 레지스트리에 승인된 것"을 대신 확인할 수도 있습니다.
</details>

### 5.3. 과제 2: 직접 바꿔 보기 (서술)

README 4단계의 두 가지 변경을 노트북에서 해 봅니다. 각 셀을 실행하기 전에 결과를 먼저 예측합니다.

(1) B의 승인 목록에 `report-agent`를 더하고 `secure`로 6건을 다시 보냅니다. `services.py`에서 `McpB`의 `allowed_clients`를 고친 것과 같습니다.

In [11]:
sv.McpB.allowed_clients = {"docs-hub", "report-agent"}
rows = run_scenarios("secure")
sv.McpB.allowed_clients = {"docs-hub"}                  # 원래대로
table(["번호", "호출", "secure", "사유"], [(i, sc["note"], r[0], r[1]) for i, (sc, r) in enumerate(zip(SCENARIOS, rows), 1)])

| 번호 | 호출 | secure | 사유 |
|---|---|---|---|
| 1 | 승인된 Agent가 A의 문서 검색 호출(정상, 1단계) | 실행 | 사내 문서 검색 실행됨(사용자: kim) |
| 2 | A가 다시 B를 부르는 고객 요약(정상, 2단계) | 실행 | MCP B 응답 ▶ 고객 3명 조회. B가 확인한 경로: kim → report-agent → docs-hub, scope=crm:read |
| 3 | 인가 서버에 등록은 됐지만 A에는 승인되지 않은 Agent | 403 거부 | 이 Server에 승인되지 않은 Client: shadow-agent |
| 4 | report-agent의 토큰만 빼내 재사용(키는 없어 소유 증명을 못 만듦) | 401 거부 | 소유 증명 없음(토큰만 가진 쪽) |
| 5 | A용 토큰으로 B를 직접 호출 | 401 거부 | 이 Server용 토큰이 아님(aud=https://docs-hub.hanbit.example/mcp) |
| 6 | B용 토큰을 받아 A를 건너뛰고 B를 직접 호출 | 실행 | 고객 3명 조회. B가 확인한 경로: kim → report-agent, scope=crm:read docs:read |

(2) A가 토큰을 교환할 때 `crm:read`가 아니라 `crm:read crm:write`를 달라고 하면 어떻게 될까요? `crm_summary`의 교환 요청 범위를 바꾼 것과 같습니다.

In [12]:
for asked in ("crm:read", "crm:read crm:write"):
    status, body = hub.exchange(t1, sv.B_RESOURCE, asked)
    print(f"요청 범위 {asked!r:22} → {status}, 받은 scope={sv.peek(body['access_token'])['scope']!r}")
print("토큰①의 scope:", repr(c1["scope"]), "/ 김대리에게 허락된 범위:", repr(sv.USERS["kim"]))

요청 범위 'crm:read'             → 200, 받은 scope='crm:read'
요청 범위 'crm:read crm:write'   → 200, 받은 scope='crm:read'
토큰①의 scope: 'crm:read docs:read' / 김대리에게 허락된 범위: 'docs:read crm:read crm:write'


1. (1)에서 6번이 실행되었다면 B를 직접 부르는 길이 열린 것입니다. 실무에서 이 승인 목록은 누가, 어떤 절차로 승인해야 할까요?
2. (2)에서 김대리에게는 `crm:write` 권한이 있는데도 A가 얻지 못한 이유는 무엇인가요? 이 성질이 없다면 어떤 사고가 날 수 있을까요?

✍️ **답 적기** (이 셀을 더블클릭해 고칩니다)

1.
2.

## 6. 정리

| 섹션 | 내용 | 진행 상태 |
| --- | --- | --- |
| 0 | 실습 준비와 실습 파일 | ✅ 완료 |
| 1 | 토큰만 확인하는 MCP Server | ✅ 완료 |
| 2 | 호출자까지 확인하는 MCP Server | ✅ 완료 |
| 3 | A가 B를 부를 때 무슨 일이 일어났나 | ✅ 완료 |
| 4 | 확인한 것과 확인하지 못한 것 | ✅ 완료 |
| 5 | 체크포인트와 과제 | ✅ 완료 |
| 6 | 정리 | <span style="color: #FF6B6B">⬅️ 진행 중</span> |

In [13]:
for s in SERVERS:
    s.shutdown()
    s.server_close()
sys.modules.pop("services", None)
shutil.rmtree(WORK, ignore_errors=True)
print("서비스 종료, 실습 폴더 삭제됨")

서비스 종료, 실습 폴더 삭제됨


---
> 9장 실습 2에서는 한 요청이 지나간 길을 Trace ID로 처음부터 끝까지 재구성합니다. 이 실습의 `act` 경로를 그 기록에 함께 남기면, 어느 Agent가 어느 MCP Server를 거쳐 왔는지까지 되짚을 수 있습니다.